In [ ]:
## Packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.linear_model import LassoCV, ElasticNetCV, Ridge, LinearRegression, RidgeCV
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.cross_decomposition import PLSRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

import joblib

In [ ]:
## Configuration machine
n_cores = joblib.cpu_count(only_physical_cores=True)
print(f'Nombre de coeurs (physiques): {n_cores}')

In [ ]:
## Rechargement des données nettoyées (issues du fichier 01)
directory_data = '../'
SORTIE = "../Travail Julie/"
df = pd.read_csv(SORTIE + 'engie_data_clean.csv', sep=';', decimal='.')

In [ ]:
## Gestion des variables
# ID : simple identifiant de ligne (lié à input_testing.csv), sans valeur prédictive -> exclu
# MAC_CODE : identifiant de la turbine (catégoriel, non numérique tel quel) -> exclu
# Date_time : horodatage -> exclu des features "brutes" (traité séparément si besoin, ex: saisonnalité)
cols_to_exclude = ['TARGET', 'MAC_CODE', 'Date_time', 'ID']
features = list(df.columns.difference(cols_to_exclude))

X = df[features]
y = df['TARGET']

In [ ]:
## Création d'un échantillon test
test_portion = 1/5
X_train_valid, X_test, y_train_valid, y_test = train_test_split(X, y, test_size=test_portion, random_state=1789)

print('Dimensions de X_train_valid :', X_train_valid.shape)
print('Dimensions de X_test        :', X_test.shape)
print('Dimensions de y_train_valid :', y_train_valid.shape)
print('Dimensions de y_test        :', y_test.shape)

In [ ]:
## Normalisation des covariables
scaler = StandardScaler(with_mean=True, with_std=True)
scaler.fit(X_train_valid)
X_train_valid_norm = scaler.transform(X_train_valid)
X_test_norm = scaler.transform(X_test)

In [ ]:
## Split apprentissage / validation
valid_portion = 1/5
X_train, X_valid, y_train, y_valid = train_test_split(X_train_valid, y_train_valid, test_size=valid_portion, random_state=1234)

print('Dimensions de X_train :', X_train.shape)
print('Dimensions de X_valid :', X_valid.shape)
print('Dimensions de X_test  :', X_test.shape)
print('Dimensions de y_train :', y_train.shape)
print('Dimensions de y_valid :', y_valid.shape)
print('Dimensions de y_test  :', y_test.shape)

In [ ]:
## Tableau de stockage des prévisions
PREV = pd.DataFrame({"Y": y_test.values})
for col in ["MCO", "lasso", "ridge", "elastic", "ridgeGS", "arbre", "foret", "pcr", "pls"]:
    PREV[col] = 0.0
# ajouter boosting et DNN

In [ ]:
## Configuration des modèles
cr = StandardScaler()
kf = KFold(n_splits=10, shuffle=True, random_state=0)
kfaxes = KFold(n_splits=4, shuffle=True, random_state=0)
nbaxes = 40

lassocv = LassoCV(cv=kf, max_iter=50000)
pipe_lassocv = Pipeline(steps=[("cr", cr), ("lassocv", lassocv)])
etape_lassocv = pipe_lassocv.named_steps["lassocv"]
lambdaoptlasso = []

enetcv = ElasticNetCV(cv=kf, max_iter=50000)
pipe_enetcv = Pipeline(steps=[("cr", cr), ("enetcv", enetcv)])

ridge = Ridge()
pipe_ridge = Pipeline(steps=[("cr", cr), ("ridge", ridge)])

acp = PCA()
reg_pcr = LinearRegression()
pipe_pcr = Pipeline(steps=[("cr", cr), ("acp", acp), ("reg", reg_pcr)])
param_grid_pcr = {"acp__n_components": list(range(1, nbaxes))}

regpls = PLSRegression()
param_grid_pls = {"n_components": list(range(1, nbaxes))}

In [ ]:
## Exécution des modèles sur échantillon d'apprentissage
Xapp  = X_train_valid
Xtest = X_test
Yapp = y_train_valid
Ytest = y_test


### MCO (toutes variables, sans sélection ni pénalité)
reg = LinearRegression()
reg.fit(Xapp, Yapp)
PREV["MCO"] = reg.predict(Xtest)

### BIC (commenté : coûteux quand le nb de features est grand)
# inst_reg_bic = ols_step_sk.LinearRegressionSelectionFeatureIC(verbose=1, crit="bic")
# reg_bic = inst_reg_bic.fit(X=Xapp, y=Yapp)
# PREV["BIC"] = reg_bic.predict(Xtest)

### AIC
# inst_reg_aic = ols_step_sk.LinearRegressionSelectionFeatureIC(verbose=1, crit="aic")
# reg_aic = inst_reg_aic.fit(X=Xapp, y=Yapp)
# PREV["AIC"] = reg_aic.predict(Xtest)

### Lasso (sélection de variables via pénalité L1)
pipe_lassocv.fit(Xapp, Yapp)
lambdaoptlasso.append(round(etape_lassocv.alpha_, 4))
PREV["lasso"] = pipe_lassocv.predict(Xtest)

### ElasticNet (compromis L1/L2)
pipe_enetcv.fit(Xapp, Yapp)
PREV["elastic"] = pipe_enetcv.predict(Xtest)

### Ridge via RidgeCV (grille calée sur le chemin lasso x 100)
alphasridge = 100 * etape_lassocv.alphas_
ridgecv = RidgeCV(cv=kf, alphas=alphasridge)
pipe_ridgecv = Pipeline(steps=[("cr", cr), ("ridgecv", ridgecv)])
pipe_ridgecv.fit(Xapp, Yapp)
PREV["ridge"] = pipe_ridgecv.predict(Xtest)

### Ridge via GridSearchCV (alternative plus flexible)
param_grid_ridge = {"ridge__alpha": alphasridge}
cv_ridge = GridSearchCV(pipe_ridge, param_grid_ridge, cv=kf,
                        scoring="neg_mean_squared_error", n_jobs=3).fit(Xapp, Yapp)
PREV["ridgeGS"] = cv_ridge.predict(Xtest)

### Arbre de décision (min 5 obs par feuille pour limiter le surapprentissage)
arbre = DecisionTreeRegressor(min_samples_leaf=5).fit(Xapp, Yapp)
PREV["arbre"] = arbre.predict(Xtest)

### Forêt aléatoire
foret = RandomForestRegressor().fit(Xapp, Yapp)
PREV["foret"] = foret.predict(Xtest)

### PCR : ACP puis régression linéaire sur les composantes
cv_pcr = GridSearchCV(pipe_pcr, param_grid_pcr, cv=kfaxes,
                      scoring="neg_mean_squared_error", n_jobs=3).fit(Xapp, Yapp)
PREV["pcr"] = cv_pcr.predict(Xtest)

### PLS : maximise la covariance entre X et Y dans l'espace réduit
cv_pls = GridSearchCV(regpls, param_grid_pls, cv=kfaxes,
                      scoring="neg_mean_squared_error", n_jobs=3).fit(Xapp, Yapp)
PREV["pls"] = cv_pls.predict(Xtest)

In [ ]:
## Comparaison des performances des modèles via MAE
prev = PREV.drop(columns=["Y"])
mae = np.round(abs(prev.sub(PREV["Y"], axis=0)).mean(), 2)
print("MAE par méthode :")
print(mae.sort_values())

print(f"PREV charge : {PREV.shape[0]} observations x {PREV.shape[1]} colonnes")
print(f"Methodes disponibles : {list(PREV.columns[1:])}")
PREV.head()

In [ ]:

## Vérification des hyperparamètres sélectionnés par validation croisée
print("--- Hyperparamètres optimaux sélectionnés ---")
print("Lasso  - lambda optimal :", lambdaoptlasso)
print("ElasticNet - alpha optimal :", round(pipe_enetcv.named_steps['enetcv'].alpha_, 4))
print("RidgeCV - alpha optimal :", round(pipe_ridgecv.named_steps['ridgecv'].alpha_, 4))
print("GridSearchCV Ridge - alpha optimal :", cv_ridge.best_params_)
print("PCR - nb composantes optimal :", cv_pcr.best_params_)
print("PLS - nb composantes optimal :", cv_pls.best_params_)

## Complément utile : R² en plus du MAE, pour une lecture plus intuitive de la performance
from sklearn.metrics import r2_score
r2_scores = {col: round(r2_score(PREV['Y'], PREV[col]), 3) for col in PREV.columns[1:10]}
print("\n--- R² par méthode ---")
print(pd.Series(r2_scores).sort_values(ascending=False))


In [ ]:
## ---------------------------------------------------------
## Analyse des résidus : écart entre production prédite et réelle
## (objectif final : détecter les anomalies de fonctionnement)
## ---------------------------------------------------------

# --- 1. Sélection du meilleur modèle selon le MAE déjà calculé ---
# (tu peux remplacer par un nom de modèle fixe si tu préfères en choisir un manuellement,
#  ex: best_model_name = "foret")
best_model_name = mae.sort_values().index[0]
print(f"Modèle retenu pour l'analyse des résidus : {best_model_name} (MAE = {mae[best_model_name]})")

# --- 2. Calcul du résidu sur l'échantillon test (hors apprentissage, donc fiable) ---
PREV['residual'] = PREV['Y'] - PREV[best_model_name]

print("\n--- Statistiques descriptives du résidu ---")
print(PREV['residual'].describe())

# --- 3. Visualisation : distribution des résidus ---
plt.figure(figsize=(8, 5))
plt.hist(PREV['residual'], bins=50, edgecolor='black', alpha=0.7)
plt.axvline(0, color='red', linestyle='--', linewidth=1)
plt.title(f"Distribution des résidus - modèle {best_model_name}")
plt.xlabel("Résidu (Y réel - Y prédit)")
plt.ylabel("Fréquence")
plt.tight_layout()
plt.show()

# --- 4. Visualisation : prédiction vs réalité ---
plt.figure(figsize=(7, 7))
plt.scatter(PREV['Y'], PREV[best_model_name], alpha=0.3, s=10)
lims = [min(PREV['Y'].min(), PREV[best_model_name].min()),
        max(PREV['Y'].max(), PREV[best_model_name].max())]
plt.plot(lims, lims, color='red', linestyle='--', linewidth=1)  # ligne y=x (prédiction parfaite)
plt.xlabel("TARGET réel")
plt.ylabel("TARGET prédit")
plt.title(f"Prédit vs Réel - modèle {best_model_name}")
plt.tight_layout()
plt.show()

# --- 5. Définition d'un seuil d'anomalie sur le résidu ---
# Approche classique : k écarts-types autour de la moyenne du résidu (ex: k=3)
k = 3
residual_mean = PREV['residual'].mean()
residual_std = PREV['residual'].std()
lower_threshold = residual_mean - k * residual_std
upper_threshold = residual_mean + k * residual_std

PREV['anomalie'] = (PREV['residual'] < lower_threshold) | (PREV['residual'] > upper_threshold)

print(f"\nSeuils d'anomalie (k={k}) : [{lower_threshold:.2f} ; {upper_threshold:.2f}]")
print(f"Nombre d'observations signalées comme anomalies : {PREV['anomalie'].sum()} / {len(PREV)}")

# --- 6. Visualisation des résidus avec seuils, colorés par statut d'anomalie ---
plt.figure(figsize=(12, 5))
colors = PREV['anomalie'].map({True: 'red', False: 'grey'})
plt.scatter(range(len(PREV)), PREV['residual'], c=colors, alpha=0.5, s=10)
plt.axhline(upper_threshold, color='black', linestyle='--', linewidth=0.8)
plt.axhline(lower_threshold, color='black', linestyle='--', linewidth=0.8)
plt.axhline(0, color='blue', linewidth=0.5)
plt.title(f"Résidus sur l'échantillon test - anomalies en rouge (k={k} écarts-types)")
plt.xlabel("Index de l'observation (échantillon test)")
plt.ylabel("Résidu")
plt.tight_layout()
plt.show()

# --- 7. Récupération des identifiants d'origine pour investigation ---
# X_test.index correspond aux index d'origine dans df -> permet de retrouver ID/MAC_CODE/Date_time
id_cols_available = [c for c in ['ID', 'MAC_CODE', 'Date_time'] if c in df.columns]
PREV_detail = PREV.copy()
PREV_detail.index = X_test.index  # réaligne PREV sur les vrais index de df
PREV_detail = df.loc[X_test.index, id_cols_available].join(PREV_detail)

print(f"\n--- Détail des {PREV_detail['anomalie'].sum()} observations anormales ---")
display(PREV_detail[PREV_detail['anomalie']].sort_values('residual', key=abs, ascending=False))


In [ ]:

## Alternative : seuil par percentile (ex. 1% des observations les plus extrêmes dans chaque sens)
p_low, p_high = 0.5, 99.5  # à ajuster selon le taux d'anomalies que tu juges réaliste
lower_threshold_pct = np.percentile(PREV['residual'], p_low)
upper_threshold_pct = np.percentile(PREV['residual'], p_high)

PREV['anomalie_pct'] = (PREV['residual'] < lower_threshold_pct) | (PREV['residual'] > upper_threshold_pct)
print(f"Seuil percentile [{p_low}-{p_high}] : [{lower_threshold_pct:.2f} ; {upper_threshold_pct:.2f}] "
      f"-> {PREV['anomalie_pct'].sum()} anomalies")


In [ ]:

## Remplacement du résidu relatif classique par le SMAPE (borné entre -200% et +200%, jamais infini)
PREV['smape_%'] = 200 * (PREV['residual']) / (PREV['Y'].abs() + PREV['foret'].abs())

print("--- Statistiques du SMAPE (%) ---")
print(PREV['smape_%'].describe())

top_smape = PREV.reindex(PREV['smape_%'].abs().sort_values(ascending=False).index).head(10)
print("\n--- Top 10 par SMAPE ---")
print(top_smape[['Y', 'foret', 'residual', 'smape_%']])



In [ ]:

## ---------------------------------------------------------
## Complément 3 : résidus dans le temps réel (Date_time), pas l'ordre aléatoire du test
## ---------------------------------------------------------

# Rappel : l'index de PREV correspond à l'ordre issu du train_test_split (aléatoire),
# donc PREV_detail (construit précédemment) doit être trié par la vraie variable temporelle
PREV_detail_sorted = PREV_detail.sort_values('Date_time')

plt.figure(figsize=(14, 5))
colors = PREV_detail_sorted['anomalie'].map({True: 'red', False: 'grey'})
plt.scatter(PREV_detail_sorted['Date_time'], PREV_detail_sorted['residual'], c=colors, alpha=0.5, s=10)
plt.axhline(upper_threshold, color='black', linestyle='--', linewidth=0.8)
plt.axhline(lower_threshold, color='black', linestyle='--', linewidth=0.8)
plt.axhline(0, color='blue', linewidth=0.5)
plt.title("Résidus triés par Date_time réel - anomalies en rouge")
plt.xlabel("Date_time (horodatage, entier = pas de 10 minutes)")
plt.ylabel("Résidu")
plt.tight_layout()
plt.show()


In [ ]:

## Zoom sur le tout début de la période temporelle
early_period = PREV_detail_sorted[PREV_detail_sorted['Date_time'] < 20000]
print(f"Nombre d'anomalies dans les 20 000 premiers pas de temps : {early_period['anomalie'].sum()} / {len(early_period)}")
print(f"Proportion : {100*early_period['anomalie'].sum()/len(early_period):.1f}%")

# Comparaison avec le reste de la période
rest_period = PREV_detail_sorted[PREV_detail_sorted['Date_time'] >= 20000]
print(f"Proportion sur le reste de la période : {100*rest_period['anomalie'].sum()/len(rest_period):.1f}%")
